# Analyze checkpoints: embedding vs effect & MMD along training

This notebook analyzes results from `train_and_predict.py` run with `--save-all-checkpoints`:

1. **Embedding similarity vs effect similarity**: Do drugs with more similar embeddings have more similar effects (MMD between targets)?
   - All pairs (same cell_line)
   - Holdout vs train pairs only

2. **MMD vs control MMD along training**: How do MMD(pred, target) and MMD(pred, control) evolve across checkpoint steps?
   - Per-condition trajectories
   - Sum over conditions
   - Scatter to see if all conditions are optimized equally

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

from scaleflow.metrics._metrics import compute_scalar_mmd

/home/icb/xiaotong.fu/miniconda3/envs/pancellflow/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths

In [2]:
# Output directory from train_and_predict.py (contains predictions.h5ad, mmd_along_training.csv, checkpoints/)
OUTPUT_DIR = Path("/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/outputs/train_and_predict/random_embeddings_film_otfm_unipert")

# Path to original Tahoe adata with drug embeddings (same as in train_and_predict.py)
ORIGINAL_ADATA_PATH = Path("/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/unipert/tahoe_a549_w_se.h5ad")

# Must match train_and_predict.py --seed (default 42) so the train/holdout split is identical
SEED = 42

PREDICTIONS_PATH = OUTPUT_DIR / "predictions.h5ad"
MMD_CSV_PATH = OUTPUT_DIR / "mmd_along_training.csv"

## 0. Generate predictions.h5ad and mmd_along_training.csv from checkpoints (ad hoc)

If you have **checkpoints** but never ran the prediction block (or the run didn't save `predictions.h5ad` / `mmd_along_training.csv`), run the cells below to build them from checkpoint files. This replicates the data pipeline and model setup from `train_and_predict.py`, then runs predictions at each checkpoint and saves the CSV and one predictions file (using the checkpoint with lowest mean MMD).

In [3]:
# Set to True to generate predictions.h5ad and mmd_along_training.csv from OUTPUT_DIR/checkpoints/*.pkl
# when those files are missing (or set True to always regenerate).
GENERATE_FROM_CKPT = not (PREDICTIONS_PATH.exists() and MMD_CSV_PATH.exists())
if GENERATE_FROM_CKPT:
    print("Will generate predictions and MMD CSV from checkpoints.")
else:
    print("predictions.h5ad and mmd_along_training.csv already exist; skip to section 1.")

Will generate predictions and MMD CSV from checkpoints.


In [14]:
if GENERATE_FROM_CKPT:
    import cloudpickle
    import optax
    import anndata as ad
    from scaleflow.data import AnnDataLocation, DataManager
    from scaleflow.data._dataloader import InMemorySampler, ValidationSampler
    from scaleflow.model import ScaleFlow

    # Must match train_and_predict.py (SEED is set in Paths cell above)
    OBSM_KEY = "X_state"
    SOLVER_NAME = "otfm"
    EXPERIMENT_TYPE = "condition_subsampling"  # or "full"
    N_HOLDOUT = 60

    def _holdout_conditions(adata, n_holdout, seed):
        rng = np.random.default_rng(seed)
        condition_list = adata.obs.groupby(["drug_0", "drug_1"], observed=True).size().index.tolist()
        control = ("control", "control")
        non_control = [c for c in condition_list if c != control]
        holdout_idx = rng.choice(len(non_control), size=n_holdout, replace=False)
        holdout = [non_control[i] for i in holdout_idx]
        holdout_mask = adata.obs.set_index(["drug_0", "drug_1"]).index.isin(holdout)
        control_mask = adata.obs["control"]
        train_adata = adata[~holdout_mask].copy()
        return train_adata

    adata = sc.read_h5ad(ORIGINAL_ADATA_PATH)
    adata.obs["control"] = (adata.obs["drug_0"] == "control") & (adata.obs["drug_1"] == "control")
    adata_full = adata
    if EXPERIMENT_TYPE == "condition_subsampling":
        adata_train = _holdout_conditions(adata, N_HOLDOUT, SEED)
    else:
        adata_train = adata

    adl = AnnDataLocation()
    data_manager = DataManager(
        dist_flag_key="control",
        src_dist_keys=["cell_line"],
        tgt_dist_keys=["drug_0"],
        rep_keys={"cell_line": "cell_line_embeddings", "drug_0": "drug_0_embeddings"},
        data_location=adl.obsm[OBSM_KEY],
    )
    gd = data_manager.prepare_data(adata_train)
    train_split = gd
    sampler = InMemorySampler(train_split, np.random.default_rng(SEED), batch_size=512)
    sampler.init_sampler()
    sample_batch = sampler.sample()
    print("Data and sample_batch ready.")

Data and sample_batch ready.


In [15]:
if GENERATE_FROM_CKPT:
    sf = ScaleFlow(solver=SOLVER_NAME)
    decoder_dims = hidden_dims = time_encoder_dims = (256, 256, 256)
    sf.prepare_model(
        sample_batch=sample_batch,
        max_combination_length=2,
        conditioning="film",
        hidden_dims=hidden_dims,
        hidden_dropout=0.1,
        time_encoder_dims=time_encoder_dims,
        time_encoder_dropout=0.1,
        condition_embedding_dim=256,
        cond_output_dropout=0.1,
        decoder_dims=decoder_dims,
        optimizer=optax.MultiSteps(optax.adamw(learning_rate=1e-4), 20),
    )
    train_drugs = set(adata_train.obs.groupby("drug_0", observed=True).groups.keys())
    gd_full = data_manager.prepare_data(adata_full)
    full_sampler = ValidationSampler(gd_full, n_conditions_on_log_iteration=None, n_conditions_on_train_end=None, seed=SEED)
    if not full_sampler.initialized:
        full_sampler.init_sampler()
    annotation = full_sampler._data.annotation
    tgt_labels = annotation.tgt_dist_idx_to_labels
    src_labels = annotation.src_dist_idx_to_labels
    tgt_to_src = full_sampler._tgt_to_src
    full_gd_data = full_sampler._data.data
    all_tgt_indices = list(full_gd_data.conditions.keys())
    src_dict = {}
    condition_dict = {}
    target_dict = {}
    for tgt_idx in all_tgt_indices:
        src_idx = tgt_to_src.get(tgt_idx)
        if src_idx is None:
            continue
        drug = tgt_labels[tgt_idx][0] if tgt_idx in tgt_labels else str(tgt_idx)
        cell_line = src_labels[src_idx][0] if src_idx in src_labels else str(src_idx)
        key = f"{drug}|{cell_line}"
        src_dict[key] = full_gd_data.src_data[src_idx]
        condition_dict[key] = full_gd_data.conditions[tgt_idx]
        target_dict[key] = full_gd_data.tgt_data[tgt_idx]
    print(f"Model ready. {len(src_dict)} conditions for prediction.")

Model ready. 367 conditions for prediction.


In [16]:
if GENERATE_FROM_CKPT:
    ckpt_dir = OUTPUT_DIR / "checkpoints"
    # Only these validation steps for MMD along training (skip missing files)
    STEPS_FOR_MMD = [1, 4, 8, 16, 32]
    best_params_path = OUTPUT_DIR / "best_params.pkl"
    if not best_params_path.exists():
        raise FileNotFoundError(f"Best params not found: {best_params_path}")
    mmd_rows = []
    for step in STEPS_FOR_MMD:
        p = ckpt_dir / f"checkpoint_{step:04d}.pkl"
        if not p.exists():
            print(f"Checkpoint {p} not found, skipping step {step}")
            continue
        with open(p, "rb") as f:
            step_params = cloudpickle.load(f)
        sf._solver.vf_state = sf._solver.vf_state.replace(params=step_params)
        sf._solver.vf_state_inference = sf._solver.vf_state_inference.replace(params=step_params)
        for key in src_dict:
            drug, cell_line = key.split("|", 1)
            pred = np.array(sf._solver.predict(src_dict[key], condition_dict[key]))
            tgt = np.array(target_dict[key])
            src_arr = np.array(src_dict[key])
            mmd_pred_tgt = float(compute_scalar_mmd(pred, tgt))
            mmd_pred_control = float(compute_scalar_mmd(pred, src_arr))
            mmd_rows.append({
                "checkpoint_step": step, "drug_0": drug, "cell_line": cell_line, "condition_key": key,
                "mmd_pred_tgt": mmd_pred_tgt, "mmd_pred_control": mmd_pred_control,
                "in_train": drug in train_drugs,
            })
    mmd_df = pd.DataFrame(mmd_rows)
    mmd_df.to_csv(MMD_CSV_PATH, index=False)
    print(f"Saved {MMD_CSV_PATH} ({len(mmd_df)} rows)")
    


KeyboardInterrupt: 

In [ ]:
p

In [ ]:
    # predictions.h5ad from best params (saved by train_and_predict.py), not from step checkpoints
    with open(best_params_path, "rb") as f:
        best_params = cloudpickle.load(f)
    sf._solver.vf_state = sf._solver.vf_state.replace(params=best_params)
    sf._solver.vf_state_inference = sf._solver.vf_state_inference.replace(params=best_params)
    rows_X, rows_obs = [], []
    for key in src_dict:
        drug, cell_line = key.split("|", 1)
        pred = np.array(sf._solver.predict(src_dict[key], condition_dict[key]))
        tgt = np.array(target_dict[key])
        src_arr = np.array(src_dict[key])
        in_train = drug in train_drugs
        for arr, split in [(pred, "prediction"), (tgt, "target"), (src_arr, "source")]:
            rows_X.append(arr)
            rows_obs.append(pd.DataFrame({"drug_0": drug, "cell_line": cell_line, "split": split, "in_train": in_train}, index=range(len(arr))))
    obs = pd.concat(rows_obs, ignore_index=True)
    X = np.concatenate(rows_X, axis=0)
    adata_pred = ad.AnnData(X=X, obs=obs)
    adata_pred.uns["obsm_key"] = OBSM_KEY
    adata_pred.write_h5ad(PREDICTIONS_PATH)
    print(f"Saved {PREDICTIONS_PATH} (from best_params.pkl)")

## 1. Embedding similarity vs effect similarity

- **True similarity**: Embedding distance vs effect distance (MMD between **true** target distributions). Uses **original adata only** — no predicted data.
- **Pred–target vs pred–control**: MMD(pred, target) vs MMD(pred, true control) per condition (from `mmd_along_training.csv`; does not require `predictions.h5ad`).

In [ ]:
def embedding_vs_effect_similarity(adata_orig, obsm_key="X_state", in_train_per_drug=None):
    """Compute embedding distance and effect distance (MMD) for pairs of conditions (same cell_line). Uses only true data from adata_orig."""
    emb_key = "drug_0_embeddings"
    if emb_key not in adata_orig.uns:
        raise KeyError(f"Original adata must have uns['{emb_key}']")
    emb_dict = adata_orig.uns[emb_key]

    def emb_vec(drug):
        return np.asarray(emb_dict.get(drug, np.nan)).ravel()

    def emb_dist(d1, d2):
        v1, v2 = emb_vec(d1), emb_vec(d2)
        if np.any(np.isnan(v1)) or np.any(np.isnan(v2)):
            return np.nan
        return float(np.linalg.norm(v1 - v2))

    groups = adata_pred.obs.groupby(["drug_0", "cell_line"], observed=True)
    condition_keys = list(groups.indices.keys())
    tgt_arrays = {}
    for (drug, cell_line) in condition_keys:
        mask = (
            (adata_pred.obs["drug_0"] == drug)
            & (adata_pred.obs["cell_line"] == cell_line)
            & (adata_pred.obs["split"] == "target")
        )
        if mask.sum() == 0:
            continue
        tgt_arrays[(drug, cell_line)] = np.asarray(adata_pred[mask].X)

    rows = []
    cell_lines = list(set(c for _, c in tgt_arrays.keys()))
    for cell_line in cell_lines:
        drugs = [d for (d, c) in tgt_arrays.keys() if c == cell_line]
        for i in range(len(drugs)):
            for j in range(i + 1, len(drugs)):
                d1, d2 = drugs[i], drugs[j]
                e_dist = emb_dist(d1, d2)
                if np.isnan(e_dist):
                    continue
                t1 = tgt_arrays[(d1, cell_line)]
                t2 = tgt_arrays[(d2, cell_line)]
                effect_dist = float(compute_scalar_mmd(t1, t2))
                in_train_1 = in_train_per_drug.get(d1, None) if in_train_per_drug else None
                in_train_2 = in_train_per_drug.get(d2, None) if in_train_per_drug else None
                holdout_train_pair = (
                    in_train_1 is not None
                    and in_train_2 is not None
                    and in_train_1 != in_train_2
                )
                rows.append({
                    "drug_1": d1,
                    "drug_2": d2,
                    "cell_line": cell_line,
                    "embedding_dist": e_dist,
                    "effect_dist_mmd": effect_dist,
                    "holdout_train_pair": holdout_train_pair,
                })
    return pd.DataFrame(rows)

In [ ]:
adata_pred = sc.read_h5ad(PREDICTIONS_PATH)
adata_orig = sc.read_h5ad(ORIGINAL_ADATA_PATH)

print(f"Loaded {adata_pred.n_obs} cells, {adata_pred.obs.groupby(['drug_0', 'cell_line'], observed=True).ngroups} conditions")

# Build in_train set from predictions
_in_train = {}
for _, row in adata_pred.obs.drop_duplicates("drug_0").iterrows():
    _in_train[row["drug_0"]] = bool(row.get("in_train", False))

df_emb_eff = embedding_vs_effect_similarity(adata_pred, adata_orig, in_train_per_drug=_in_train)
print(f"Computed {len(df_emb_eff)} drug pairs")
df_emb_eff.head(10)

### 1b. Pred–target vs pred–control (from MMD CSV)

Per-condition MMD(pred, target) vs MMD(pred, true control). No `predictions.h5ad` needed.

In [ ]:
# Pred–target vs pred–control: use one checkpoint step (e.g. last in CSV)
if MMD_CSV_PATH.exists():
    _mmd = pd.read_csv(MMD_CSV_PATH)
    _steps = sorted(_mmd["checkpoint_step"].unique())
    _step = _steps[-1] if _steps else None
    if _step is not None:
        _df = _mmd[_mmd["checkpoint_step"] == _step]
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        for in_tr, label in [(True, "train"), (False, "holdout")]:
            sub = _df[_df["in_train"] == in_tr]
            if len(sub) > 0:
                ax.scatter(sub["mmd_pred_control"], sub["mmd_pred_tgt"], alpha=0.6, label=label, s=20)
        ax.set_xlabel("MMD(pred, true control)")
        ax.set_ylabel("MMD(pred, target)")
        ax.set_title(f"Step {_step} (n={len(_df)} conditions)")
        ax.legend()
        ax.spines[["top", "right"]].set_visible(False)
        plt.tight_layout()
        plt.show()
else:
    print(f"MMD CSV not found: {MMD_CSV_PATH}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# All pairs
df_all = df_emb_eff
ax = axes[0]
ax.scatter(df_all["embedding_dist"], df_all["effect_dist_mmd"], alpha=0.4, s=15)
r_p, p_p = pearsonr(df_all["embedding_dist"], df_all["effect_dist_mmd"])
r_s, p_s = spearmanr(df_all["embedding_dist"], df_all["effect_dist_mmd"])
ax.set_xlabel("Drug embedding distance (L2)")
ax.set_ylabel("Effect distance (MMD between targets)")
ax.set_title(f"All pairs (n={len(df_all)})\nPearson r={r_p:.3f} (p={p_p:.2e}), Spearman ρ={r_s:.3f} (p={p_s:.2e})")
ax.spines[["top", "right"]].set_visible(False)

# Holdout vs train only
df_ht = df_emb_eff[df_emb_eff["holdout_train_pair"]]
ax = axes[1]
if len(df_ht) > 0:
    ax.scatter(df_ht["embedding_dist"], df_ht["effect_dist_mmd"], alpha=0.6, s=25)
    r_p, p_p = pearsonr(df_ht["embedding_dist"], df_ht["effect_dist_mmd"])
    r_s, p_s = spearmanr(df_ht["embedding_dist"], df_ht["effect_dist_mmd"])
    ax.set_title(f"Holdout–train pairs only (n={len(df_ht)})\nPearson r={r_p:.3f} (p={p_p:.2e}), Spearman ρ={r_s:.3f} (p={p_s:.2e})")
else:
    ax.set_title("Holdout–train pairs only (n=0)")
ax.set_xlabel("Drug embedding distance (L2)")
ax.set_ylabel("Effect distance (MMD between targets)")
ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Embedding similarity vs effect similarity", y=1.02)
plt.tight_layout()
plt.show()

## 2. MMD vs control MMD along training

In [ ]:
if not MMD_CSV_PATH.exists():
    print(f"Not found: {MMD_CSV_PATH}")
    print("Run train_and_predict.py with --save-all-checkpoints and --checkpoint-steps.")
else:
    mmd_along = pd.read_csv(MMD_CSV_PATH)
    steps = sorted(mmd_along["checkpoint_step"].unique())
    print(f"Loaded {len(mmd_along)} rows, checkpoint steps: {steps}")

In [ ]:
if MMD_CSV_PATH.exists():
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # 1) Per-condition MMD and control MMD over steps
    ax = axes[0, 0]
    for key, grp in mmd_along.groupby("condition_key"):
        ax.plot(grp["checkpoint_step"], grp["mmd_pred_tgt"], color="C0", alpha=0.2, linewidth=0.8)
    mean_mmd = mmd_along.groupby("checkpoint_step")["mmd_pred_tgt"].mean()
    ax.plot(mean_mmd.index, mean_mmd.values, color="C0", linewidth=2, label="Mean MMD(pred,tgt)")
    ax.set_xlabel("Checkpoint step")
    ax.set_ylabel("MMD(pred, target)")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("MMD(pred, target) per condition along training")

    ax = axes[0, 1]
    for key, grp in mmd_along.groupby("condition_key"):
        ax.plot(grp["checkpoint_step"], grp["mmd_pred_control"], color="C1", alpha=0.2, linewidth=0.8)
    mean_ctrl = mmd_along.groupby("checkpoint_step")["mmd_pred_control"].mean()
    ax.plot(mean_ctrl.index, mean_ctrl.values, color="C1", linewidth=2, label="Mean MMD(pred,control)")
    ax.set_xlabel("Checkpoint step")
    ax.set_ylabel("MMD(pred, control)")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("MMD(pred, control) per condition along training")

    # 2) Sum of MMD and sum of control MMD over steps
    ax = axes[1, 0]
    sum_mmd = mmd_along.groupby("checkpoint_step")["mmd_pred_tgt"].sum()
    sum_ctrl = mmd_along.groupby("checkpoint_step")["mmd_pred_control"].sum()
    ax.plot(sum_mmd.index, sum_mmd.values, "o-", label="Sum MMD(pred,tgt)")
    ax.plot(sum_ctrl.index, sum_ctrl.values, "s-", label="Sum MMD(pred,control)")
    ax.set_xlabel("Checkpoint step")
    ax.set_ylabel("Sum over conditions")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("Sum of MMD and control MMD per step")

    # 3) Scatter: MMD vs control MMD at each step
    ax = axes[1, 1]
    for step in steps:
        sub = mmd_along[mmd_along["checkpoint_step"] == step]
        ax.scatter(sub["mmd_pred_control"], sub["mmd_pred_tgt"], alpha=0.5, s=20, label=f"Step {step}")
    ax.set_xlabel("MMD(pred, control)")
    ax.set_ylabel("MMD(pred, target)")
    ax.legend(bbox_to_anchor=(1.02, 1), fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("MMD vs control MMD per condition (by step)")

    fig.suptitle("MMD and control MMD along training", y=1.02)
    plt.tight_layout()
    plt.show()